# Aula 12 — Laboratório de SVM, margens e kernels

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/03-machine-learning/notebooks/12-svm-kernels-laboratorio.ipynb)

Verificaremos margem, hinge loss, kernel RBF, seleção sem leakage, forma dual e invariância a unidades.

**Protocolo:** reservar teste → selecionar nos folds de desenvolvimento → congelar pipeline → avaliar uma vez.

## Ambiente

Dependências mínimas: Python 3.10; NumPy 1.24; pandas 2.0; Matplotlib 3.7; scikit-learn 1.3. Os dados são sintéticos, locais e sem credenciais.

In [ ]:
import os, warnings
os.environ["MPLBACKEND"] = "Agg"
warnings.filterwarnings("error")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.datasets import make_blobs, make_moons
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
SEED = 20260908
print({"numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__})

## 1. Margem linear calculável

Negativos em −2 e −1; positivos em 1 e 2. Esperamos fronteira em zero, suporte em ±1 e margem total 2.

In [ ]:
X_toy = np.array([[-2.], [-1.], [1.], [2.]])
y_toy = np.array([-1, -1, 1, 1])
toy = SVC(kernel="linear", C=1e6).fit(X_toy, y_toy)
w = toy.coef_.ravel(); b = float(toy.intercept_[0])
largura = 2 / np.linalg.norm(w)
assert np.allclose(np.sort(toy.support_vectors_.ravel()), [-1, 1], atol=1e-5)
assert abs(b) < 1e-8 and np.isclose(largura, 2, atol=1e-5)
print(f"w={w[0]:.9f} | b={b:.9f} | margem={largura:.9f}")
print("support vectors:", toy.support_vectors_.ravel().tolist())

## 2. Hinge loss e slack

Para margem funcional \(m=yf(x)\), a perda é \(\max(0,1-m)\).

In [ ]:
margens = np.array([1.5, 1.0, 0.6, 0.0, -0.4])
hinge = np.maximum(0, 1 - margens)
assert np.allclose(hinge, [0, 0, .4, 1, 1.4])
print(pd.DataFrame({"margem": margens, "hinge_loss": hinge}).to_string(index=False))

## 3. Efeito de C

Mantemos dados e escala fixos e observamos margem, ajuste e quantidade de vetores de suporte.

In [ ]:
X_overlap, y_overlap = make_blobs(
    n_samples=360, centers=[(-1, 0), (1, 0)], cluster_std=1.25, random_state=SEED
)
rows = []
for C in [.01, .1, 1, 10, 100]:
    p = Pipeline([("scale", StandardScaler()), ("svc", SVC(kernel="linear", C=C))]).fit(X_overlap, y_overlap)
    svc = p.named_steps["svc"]
    rows.append({"C": C, "acc_treino": p.score(X_overlap, y_overlap),
                 "n_support": int(svc.n_support_.sum()),
                 "margem_padronizada": 2 / np.linalg.norm(svc.coef_)})
efeito_c = pd.DataFrame(rows)
assert efeito_c.n_support.between(2, len(y_overlap)).all()
print(efeito_c.round(6).to_string(index=False))

\(C\) muda o compromisso entre norma e violações. Acurácia de treino não escolhe regularização.

## 4. RBF manual

Conferimos \(K(x,z)=\exp(-\gamma\lVert x-zVert^2)\) contra a biblioteca.

In [ ]:
x = np.array([[0., 0.]]); z = np.array([[1., 1.]])
gamma_demo = .5
dist2 = float(np.sum((x-z)**2))
k_manual = float(np.exp(-gamma_demo * dist2))
k_lib = float(rbf_kernel(x, z, gamma=gamma_demo)[0, 0])
assert np.isclose(k_manual, np.exp(-1)) and np.isclose(k_manual, k_lib)
print(f"distância²={dist2:.1f} | manual={k_manual:.9f} | biblioteca={k_lib:.9f}")

## 5. Dados e teste lacrado

`make_moons` exige fronteira curva. Índices explícitos comprovam partições disjuntas.

In [ ]:
X, y = make_moons(n_samples=900, noise=.25, random_state=SEED)
ids = np.arange(len(y))
X_dev, X_test, y_dev, y_test, id_dev, id_test = train_test_split(
    X, y, ids, test_size=.25, stratify=y, random_state=SEED
)
assert X.shape == (900, 2) and set(id_dev).isdisjoint(id_test)
print({"desenvolvimento": len(y_dev), "teste": len(y_test), "atributos": X.shape[1]})

## 6. Seleção linear × RBF

O scaler é ajustado dentro de cada fold. A grade é pequena e logarítmica; o teste permanece fechado.

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
pipe = Pipeline([("scale", StandardScaler()), ("svc", SVC(probability=False))])
grade = [
    {"svc__kernel": ["linear"], "svc__C": [.1, 1, 10]},
    {"svc__kernel": ["rbf"], "svc__C": [.1, 1, 10, 100],
     "svc__gamma": [.01, .1, 1, 10]},
]
busca = GridSearchCV(pipe, grade, scoring="accuracy", cv=cv,
                     return_train_score=True, n_jobs=1).fit(X_dev, y_dev)
cvdf = pd.DataFrame(busca.cv_results_)
cols = ["param_svc__kernel", "param_svc__C", "param_svc__gamma",
        "mean_train_score", "mean_test_score", "std_test_score", "rank_test_score"]
print("melhores parâmetros:", busca.best_params_)
print(f"melhor CV: {busca.best_score_:.6f}")
print(cvdf.sort_values("rank_test_score")[cols].head(8).round(6).to_string(index=False))

## 7. Avaliação final única

`best_estimator_` foi reajustado no desenvolvimento. Agora abrimos o teste uma vez e comparamos ao baseline.

In [ ]:
baseline = DummyClassifier(strategy="prior").fit(X_dev, y_dev)
acc_base = accuracy_score(y_test, baseline.predict(X_test))
pred = busca.best_estimator_.predict(X_test)
acc_svm = accuracy_score(y_test, pred)
svc_final = busca.best_estimator_.named_steps["svc"]
n_support = int(svc_final.n_support_.sum())
assert acc_svm > acc_base + .25 and n_support < len(y_dev)
print(f"baseline={acc_base:.6f} | SVM={acc_svm:.6f}")
print(f"support vectors={n_support}/{len(y_dev)} ({n_support/len(y_dev):.2%})")

## 8. Reconstrução dual

No binário, `dual_coef_` já armazena coeficientes assinados. Recalculamos os scores de 25 exemplos.

In [ ]:
Xq = busca.best_estimator_.named_steps["scale"].transform(X_test[:25])
sv = svc_final.support_vectors_
if svc_final.kernel == "rbf":
    K = rbf_kernel(sv, Xq, gamma=svc_final._gamma)
elif svc_final.kernel == "linear":
    K = sv @ Xq.T
else:
    raise AssertionError("kernel inesperado")
manual = svc_final.dual_coef_.ravel() @ K + svc_final.intercept_[0]
api = svc_final.decision_function(Xq)
erro_dual = float(np.max(np.abs(manual-api)))
assert erro_dual < 1e-10
print(f"kernel={svc_final.kernel} | erro máximo dual={erro_dual:.3e}")

## 9. Troca de unidades

Multiplicamos a segunda feature por mil. Com padronização, a decisão deve permanecer; sem ela, pode mudar.

In [ ]:
Xd_u = X_dev.copy(); Xd_u[:, 1] *= 1000
Xt_u = X_test.copy(); Xt_u[:, 1] *= 1000
p1 = clone(busca.best_estimator_).fit(X_dev, y_dev)
p2 = clone(busca.best_estimator_).fit(Xd_u, y_dev)
acordo_pipe = np.mean(p1.predict(X_test) == p2.predict(Xt_u))
params = {k.replace("svc__", ""): v for k, v in busca.best_params_.items()}
r1 = SVC(**params).fit(X_dev, y_dev)
r2 = SVC(**params).fit(Xd_u, y_dev)
acordo_raw = np.mean(r1.predict(X_test) == r2.predict(Xt_u))
assert np.isclose(acordo_pipe, 1) and acordo_raw < acordo_pipe
print(f"acordo com pipeline={acordo_pipe:.6f}")
print(f"acordo sem escala={acordo_raw:.6f}")

A unidade não mudou o fenômeno. Divergência sem escala é falha do protocolo, não informação nova.

## 10. Fronteiras de decisão

In [ ]:
def fronteira(ax, est, X, y, titulo):
    a = np.linspace(X[:,0].min()-.5, X[:,0].max()+.5, 180)
    b = np.linspace(X[:,1].min()-.5, X[:,1].max()+.5, 180)
    xx, yy = np.meshgrid(a, b); grid = np.c_[xx.ravel(), yy.ravel()]
    zz = est.decision_function(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz > 0, alpha=.2, levels=1)
    ax.contour(xx, yy, zz, levels=[-1,0,1], colors=["gray","black","gray"],
               linestyles=["--","-","--"])
    ax.scatter(X[:,0], X[:,1], c=y, cmap="coolwarm", s=13, alpha=.5)
    ax.set(title=titulo, xlabel="x1", ylabel="x2")

linear = Pipeline([("scale", StandardScaler()), ("svc", SVC(kernel="linear", C=1))]).fit(X_dev, y_dev)
fig, ax = plt.subplots(1, 2, figsize=(11,4.2))
fronteira(ax[0], linear, X_dev, y_dev, "SVM linear")
fronteira(ax[1], busca.best_estimator_, X_dev, y_dev, "SVM selecionada")
plt.tight_layout(); plt.show()

**Descrição:** a linha contínua é a fronteira; linhas tracejadas marcam scores −1 e +1. O RBF pode acompanhar a curvatura das luas.

## 11. Verificações finais

In [ ]:
assert len(busca.best_estimator_.named_steps["scale"].mean_) == 2
assert svc_final.probability is False
assert np.isfinite(busca.best_estimator_.decision_function(X_test)).all()
assert cvdf.mean_test_score.notna().all()
print("Todas as verificações passaram.")
print(f"seed={SEED} | CV={busca.best_score_:.6f} | teste={acc_svm:.6f}")
print(f"support={n_support} | erro dual={erro_dual:.3e}")

## Conclusões

- A margem do caso simples foi recuperada numericamente.
- \(C\) alterou margem e número de vetores de suporte.
- O RBF manual coincidiu com a biblioteca.
- A seleção não consultou o teste.
- A soma dual reproduziu `decision_function`.
- O pipeline preservou a decisão após troca de unidades.

Na próxima aula, estudaremos o significado e o custo das métricas de regressão.